# 🐷 EVA02-Large K-Fold Training
## Pig Posture Classification with OOF Predictions & TTA

**⚠️ CHANGE `FOLD_ID` BELOW TO 0, 2, or 3 ONLY** (skip 1 & 4)

### Key Features:
- K-Fold camera-based splits (5 folds, 8 cameras)
- Layer-wise Learning Rate Decay (LLRD)
- Exponential Moving Average (EMA) smoothing
- BF16 AMP mixed precision training
- OOF validation probabilities for stacking
- Test-Time Augmentation (TTA) with class swap
- Image caching (10x faster data loading)
- Early stopping with best model checkpoint

### Hyperparameters:
- Model: EVA02-Large @ 448px (304M params)
- LR: 2e-5, LLRD: 0.85, WD: 0.05
- Batch: 32, Epochs: 30, EMA decay: 0.9999
- Label smoothing: 0.2, Drop path: 0.15

## Setup & Dependencies

In [ ]:
# =====================================================
# >>>>>>>  CHANGE THIS FOR EACH KAGGLE ACCOUNT  <<<<<<<
FOLD_ID = 0   # Set to 0, 2, or 3 ONLY
# =====================================================

# Install required packages
try:
    import timm
    import albumentations
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'timm', 'albumentations'])

In [ ]:
# Import all required libraries
import os, random, time, gc, copy, warnings, json
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

## K-Fold Configuration

In [ ]:
# K-Fold camera-based splits (8 cameras total, 3 are test domain)
FOLD_CAMERA_GROUPS = {
    0: ['pen1_tur_cam1', 'pen1_orb_cam2'],       # ~3,176 val (test cam + medium)
    1: ['pen2_tur_cam2', 'pen2_orb_cam1'],       # ~5,830 val (test cam + medium) - SKIP
    2: ['pen2_orb_cam2', 'pen1_orb_cam1'],       # ~1,564 val (test cam + light)
    3: ['pen1_tur_cam2'],                        # ~16,388 val (heavy training cam)
    4: ['pen2_tur_cam1'],                        # ~19,426 val (heavy training cam) - SKIP
}

CLASS_NAMES = {
    0: 'Lateral_lying_left', 1: 'Lateral_lying_right',
    2: 'Sitting', 3: 'Standing', 4: 'Sternal_lying',
}

## Device Detection

In [ ]:
# Detect CUDA and mixed precision support
def get_device():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'[DEVICE] CUDA: {name} ({vram:.1f} GB)')
        bf16_ok = torch.cuda.is_bf16_supported()
        print(f'[DEVICE] bf16 supported: {bf16_ok}')
        return torch.device('cuda'), 'cuda', bf16_ok
    print('[DEVICE] CPU — will be extremely slow!')
    return torch.device('cpu'), 'cpu', False

DEVICE, DEVICE_TYPE, BF16_SUPPORTED = get_device()
AMP_DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16
print(f'Using AMP dtype: {AMP_DTYPE}')

## Configuration Class

In [ ]:
class CFG:
    seed = 42 + FOLD_ID
    
    # Paths
    data_root = '/kaggle/input/pig-posture-crops'
    crops_dir = data_root + '/crops'
    master_csv = data_root + '/master.csv'
    pretrained_weights = '/kaggle/input/pretrained-large-models/eva02_large_pretrained.pth'
    
    # Model
    model_name = 'eva02_large_patch14_448.mim_m38m_ft_in22k_in1k'
    img_size = 448
    num_classes = 5
    
    # Training
    batch_size = 32
    num_workers = 10
    use_cache = True
    epochs = 30
    lr = 2e-5
    wd = 0.05
    warmup_epochs = 3
    
    # LLRD
    layer_decay = 0.85
    num_vit_layers = 24
    
    # EMA
    ema_decay = 0.9999
    
    # Regularization
    label_smoothing = 0.2
    drop_rate = 0.0
    drop_path_rate = 0.15
    
    # Early stopping
    early_stopping_patience = 10
    time_limit_hours = 11.0
    
    # Output
    output_prefix = f'eva02_large_fold{FOLD_ID}'

print(f'Configuration for FOLD {FOLD_ID}: {CFG.output_prefix}')

In [ ]:
# Seed everything for reproducibility
def seed_everything(s):
    random.seed(s); np.random.seed(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

seed_everything(CFG.seed)
print(f'Random seeds set to: {CFG.seed}')

## Layer-wise Learning Rate Decay (LLRD)

In [ ]:
def get_vit_layer_id(name, num_layers):
    """Extract ViT layer ID from parameter name"""
    if 'patch_embed' in name or 'pos_embed' in name or 'cls_token' in name:
        return 0
    if 'blocks.' in name:
        block_id = int(name.split('blocks.')[1].split('.')[0])
        return block_id + 1
    return num_layers + 1

def make_llrd_param_groups(model, base_lr, weight_decay, layer_decay, num_layers):
    """Create parameter groups with layer-wise learning rate decay"""
    param_groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        layer_id = get_vit_layer_id(name, num_layers)
        lr_scale = layer_decay ** (num_layers + 1 - layer_id)
        is_nodecay = (param.ndim == 1 or 'bias' in name or 'norm' in name or 'ln' in name)
        wd = 0.0 if is_nodecay else weight_decay
        group_key = f'layer_{layer_id}_wd{wd > 0}'
        if group_key not in param_groups:
            param_groups[group_key] = {
                'params': [], 'lr': base_lr * lr_scale,
                'weight_decay': wd, 'layer_id': layer_id,
            }
        param_groups[group_key]['params'].append(param)

    groups = sorted(param_groups.values(), key=lambda g: g['layer_id'])
    print('\nLLRD Parameter Groups:')
    for g in groups:
        n_params = sum(p.numel() for p in g['params']) / 1e6
        print(f'  Layer {g["layer_id"]:>2d}: lr={g["lr"]:.2e}, wd={g["weight_decay"]:.3f}, params={n_params:.1f}M')
    for g in groups:
        del g['layer_id']
    return groups

## Exponential Moving Average (EMA)

In [ ]:
class ModelEMA:
    """Exponential Moving Average of model weights"""
    def __init__(self, model, decay=0.9999):
        self.module = copy.deepcopy(model)
        self.module.eval()
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        """Update EMA weights"""
        for ema_p, model_p in zip(self.module.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1.0 - self.decay)

    def state_dict(self):
        return self.module.state_dict()

## Data Augmentation Pipelines

In [ ]:
def get_train_transforms():
    """Strong augmentation for training"""
    return A.Compose([
        A.RandomResizedCrop(size=(CFG.img_size, CFG.img_size), scale=(0.8, 1.0), ratio=(0.75, 1.33)),
        A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_REFLECT_101),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.GaussNoise(p=0.2),
        A.Downscale(scale_min=0.5, scale_max=0.9, p=0.15),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.3),
        A.CLAHE(clip_limit=4.0, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_val_transforms():
    """Minimal augmentation for validation"""
    return A.Compose([
        A.Resize(CFG.img_size, CFG.img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

## Dataset Classes

In [ ]:
class PigPostureDataset(Dataset):
    """Training/validation dataset with image caching and class swap on flip"""
    def __init__(self, df, crops_dir, transforms=None, is_train=True, cache=True):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.is_train = is_train
        self.crops_dir = crops_dir
        self.cached_bytes = None

        if cache:
            print(f'  Caching {len(self.df)} images into RAM...', end=' ', flush=True)
            t0 = time.time()
            self.cached_bytes = []
            for i in range(len(self.df)):
                fpath = os.path.join(self.crops_dir, self.df.iloc[i]['crop_filename'])
                with open(fpath, 'rb') as f:
                    self.cached_bytes.append(f.read())
            size_mb = sum(len(b) for b in self.cached_bytes) / 1e6
            print(f'{size_mb:.0f} MB in {time.time()-t0:.1f}s')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.cached_bytes is not None:
            buf = np.frombuffer(self.cached_bytes[idx], dtype=np.uint8)
            image = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        else:
            image_path = os.path.join(self.crops_dir, self.df.iloc[idx]['crop_filename'])
            image = cv2.imread(image_path, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.is_train:
            label = int(self.df.iloc[idx]['class_id'])
            if random.random() < 0.5:
                image = cv2.flip(image, 1)
                if label == 0: label = 1
                elif label == 1: label = 0
            if self.transforms:
                image = self.transforms(image=image)['image']
            return image, torch.tensor(label, dtype=torch.long)
        else:
            if self.transforms:
                image = self.transforms(image=image)['image']
            return image, self.df.iloc[idx]['row_id']

In [ ]:
class PigPostureValDataset(Dataset):
    """Validation dataset that returns (image, label, row_id) for OOF"""
    def __init__(self, df, crops_dir, transforms=None, cache=True):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.crops_dir = crops_dir
        self.cached_bytes = None

        if cache:
            print(f'  Caching {len(self.df)} val images into RAM...', end=' ', flush=True)
            t0 = time.time()
            self.cached_bytes = []
            for i in range(len(self.df)):
                fpath = os.path.join(self.crops_dir, self.df.iloc[i]['crop_filename'])
                with open(fpath, 'rb') as f:
                    self.cached_bytes.append(f.read())
            size_mb = sum(len(b) for b in self.cached_bytes) / 1e6
            print(f'{size_mb:.0f} MB in {time.time()-t0:.1f}s')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.cached_bytes is not None:
            buf = np.frombuffer(self.cached_bytes[idx], dtype=np.uint8)
            image = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        else:
            image_path = os.path.join(self.crops_dir, self.df.iloc[idx]['crop_filename'])
            image = cv2.imread(image_path, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transforms:
            image = self.transforms(image=image)['image']
        label = int(self.df.iloc[idx]['class_id'])
        row_id = self.df.iloc[idx]['row_id']
        return image, label, row_id

## Model Architecture

In [ ]:
class PigPostureModel(nn.Module):
    """EVA02-Large with classification head"""
    def __init__(self, model_name=CFG.model_name, num_classes=CFG.num_classes, pretrained=True):
        super().__init__()
        local_weights = CFG.pretrained_weights
        self.backbone = timm.create_model(
            model_name, pretrained=False, num_classes=0,
            img_size=CFG.img_size, drop_path_rate=CFG.drop_path_rate,
        )
        in_features = self.backbone.num_features
        self.head = nn.Linear(in_features, num_classes)

        if pretrained and local_weights and os.path.exists(local_weights):
            print(f'Loading pretrained weights from: {local_weights}')
            full_state = torch.load(local_weights, map_location='cpu', weights_only=True)
            backbone_state = {}
            skipped = []
            for k, v in full_state.items():
                if k.startswith('head.') or k.startswith('fc_norm.'):
                    skipped.append(k)
                else:
                    backbone_state[k] = v
            msg = self.backbone.load_state_dict(backbone_state, strict=False)
            print(f'  Loaded {len(backbone_state)} keys. Skipped {len(skipped)} head keys.')
            print(f'  Missing: {len(msg.missing_keys)}, Unexpected: {len(msg.unexpected_keys)}')
        elif pretrained:
            print(f'[WARNING] Local weights not found at {local_weights}. Using random init!')

    def forward(self, x):
        return self.head(self.backbone(x))

## Data Loading & K-Fold Split

In [ ]:
# Load master CSV and create K-Fold split
print(f'[DATA] Loading: {CFG.master_csv}')
df = pd.read_csv(CFG.master_csv)
df_tp = df[df['set'].isin(['train1', 'train2'])].copy().reset_index(drop=True)
df_tp['cam_group'] = (df_tp['pen'].astype(str) + '_' + df_tp['cam_type'].astype(str) + '_' + df_tp['cam_num'].astype(str))

val_cameras = FOLD_CAMERA_GROUPS[FOLD_ID]
val_mask = df_tp['cam_group'].isin(val_cameras)
df_tr = df_tp[~val_mask].reset_index(drop=True)
df_va = df_tp[val_mask].reset_index(drop=True)

print(f'\n[FOLD {FOLD_ID}] Split:')
print(f'  Train: {len(df_tr)} images | Val: {len(df_va)} images')
print(f'  Val cameras: {val_cameras}')
print(f'  Train cameras: {sorted(df_tr["cam_group"].unique())}')
print(f'  Train dist: {dict(df_tr["class_id"].value_counts().sort_index())}')
print(f'  Val dist:   {dict(df_va["class_id"].value_counts().sort_index())}')

In [ ]:
# Create datasets and loaders
print(f'\n[DATALOADER] Creating datasets...')
tr_ds = PigPostureDataset(df_tr, CFG.crops_dir, get_train_transforms(), is_train=True, cache=CFG.use_cache)
va_ds = PigPostureDataset(df_va, CFG.crops_dir, get_val_transforms(), is_train=True, cache=CFG.use_cache)

# Weighted sampling
class_counts = df_tr['class_id'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = [class_weights[int(l)] for l in df_tr['class_id'].values]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

tr_loader = DataLoader(tr_ds, batch_size=CFG.batch_size, sampler=sampler, num_workers=CFG.num_workers, pin_memory=True, drop_last=True, persistent_workers=True)
va_loader = DataLoader(va_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True)

# Test loader
df_test = df[df['set'] == 'test'].copy().reset_index(drop=True)
if len(df_test) > 0:
    test_ds = PigPostureDataset(df_test, CFG.crops_dir, get_val_transforms(), is_train=False, cache=False)
    test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
    print(f'  Test loader: {len(test_loader)} batches')

print(f'  Train loader: {len(tr_loader)} batches of {CFG.batch_size}')
print(f'  Val loader: {len(va_loader)} batches of {CFG.batch_size}')

## Model Setup & Optimizer

In [ ]:
# Load model
print('Loading EVA02-Large (304M params)...')
model = PigPostureModel().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Total params: {total_params:.1f}M')

if DEVICE_TYPE == 'cuda':
    allocated = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {allocated:.1f} / {total:.1f} GB after model load')

In [ ]:
# Optimizer with LLRD
param_groups = make_llrd_param_groups(model, base_lr=CFG.lr, weight_decay=CFG.wd, layer_decay=CFG.layer_decay, num_layers=CFG.num_vit_layers)
optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.999), eps=1e-8)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)

# Scheduler: warmup + cosine
warmup = LinearLR(optimizer, start_factor=0.01, total_iters=CFG.warmup_epochs)
cosine = CosineAnnealingLR(optimizer, T_max=CFG.epochs - CFG.warmup_epochs, eta_min=1e-7)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[CFG.warmup_epochs])

# EMA
ema = ModelEMA(model, decay=CFG.ema_decay)
print(f'EMA initialized (decay={CFG.ema_decay})')

scaler = torch.amp.GradScaler('cuda', enabled=(AMP_DTYPE == torch.float16))

## Training Loop

In [ ]:
print(f'\n{"="*70}')
print(f'  EVA02-Large K-Fold Training -- FOLD {FOLD_ID}/4')
print(f'  Model: {CFG.model_name} @ {CFG.img_size}px')
print(f'  LR={CFG.lr}, LLRD={CFG.layer_decay}, WD={CFG.wd}')
print(f'  Label Smooth={CFG.label_smoothing}, Drop Path={CFG.drop_path_rate}')
print(f'  AMP dtype: {AMP_DTYPE}, EMA decay: {CFG.ema_decay}')
print(f'  Batch: {CFG.batch_size}, Epochs: {CFG.epochs}, Patience: {CFG.early_stopping_patience}')
print(f'{"="*70}\n')

GLOBAL_START = time.time()
TIME_LIMIT = CFG.time_limit_hours * 3600

best_f1, best_ep = 0.0, 0
epochs_no_improve = 0
early_stop_epoch = None
hist = {'tl': [], 'vl': [], 'tf': [], 'vf': [], 'ef': [], 'vp': [], 'lr': []}
history_log = []

In [ ]:
print(f'Starting Training FOLD {FOLD_ID} (patience={CFG.early_stopping_patience})...\n')

for ep in range(CFG.epochs):
    elapsed = time.time() - GLOBAL_START
    if elapsed > TIME_LIMIT:
        print(f'\n[TIME LIMIT] {CFG.time_limit_hours}h reached. Stopping.')
        break

    t0 = time.time()
    model.train()
    running_loss = 0.0
    all_p_tr, all_l_tr = [], []

    for batch_idx, (imgs, lbls) in enumerate(tqdm(tr_loader, desc=f'Ep {ep+1}/{CFG.epochs}', leave=False)):
        imgs = imgs.to(DEVICE, non_blocking=True)
        lbls = lbls.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            outs = model(imgs)
            loss = criterion(outs, lbls)

        if AMP_DTYPE == torch.float16:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        ema.update(model)
        running_loss += loss.item() * imgs.size(0)
        all_p_tr.extend(outs.argmax(1).detach().cpu().numpy())
        all_l_tr.extend(lbls.cpu().numpy())

    tl = running_loss / len(tr_ds)
    tf = f1_score(all_l_tr, all_p_tr, average='macro')
    scheduler.step()

    # ---- Validation ----
    model.eval()
    ema.module.eval()
    vl = 0.0
    all_p_va, all_l_va = [], []
    all_p_ema = []

    with torch.no_grad():
        for imgs, lbls in tqdm(va_loader, desc='Val', leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            lbls = lbls.to(DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                outs = model(imgs)
                outs_ema = ema.module(imgs)
                loss = criterion(outs, lbls)

            vl += loss.item() * imgs.size(0)
            all_p_va.extend(outs.argmax(1).cpu().numpy())
            all_p_ema.extend(outs_ema.argmax(1).cpu().numpy())
            all_l_va.extend(lbls.cpu().numpy())

    vl /= len(va_ds)
    vf = f1_score(all_l_va, all_p_va, average='macro')
    ef = f1_score(all_l_va, all_p_ema, average='macro')
    vp = f1_score(all_l_va, all_p_va, average=None, labels=list(range(CFG.num_classes)))
    ep_vp = f1_score(all_l_va, all_p_ema, average=None, labels=list(range(CFG.num_classes)))

    use_ema = ef >= vf
    best_this_epoch = ef if use_ema else vf
    best_vp = ep_vp if use_ema else vp

    lr_now = optimizer.param_groups[-1]['lr']
    hist['tl'].append(tl); hist['vl'].append(vl); hist['tf'].append(tf); hist['vf'].append(vf)
    hist['ef'].append(ef); hist['vp'].append(best_vp.tolist()); hist['lr'].append(lr_now)

    is_best = best_this_epoch > best_f1
    if is_best:
        best_f1, best_ep = best_this_epoch, ep + 1
        epochs_no_improve = 0
        save_state = ema.state_dict() if use_ema else model.state_dict()
        torch.save(save_state, f'{CFG.output_prefix}_best.pth')
    else:
        epochs_no_improve += 1

    star = ' *** BEST' if is_best else f' (no improve: {epochs_no_improve}/{CFG.early_stopping_patience})'
    ema_tag = '(EMA)' if use_ema else '(raw)'
    epoch_time = time.time() - t0
    total_min = (time.time() - GLOBAL_START) / 60

    print(f'Ep {ep+1:02d}/{CFG.epochs} [{epoch_time:.0f}s] LR:{lr_now:.1e} | TrL:{tl:.4f} TrF1:{tf:.4f} | VaL:{vl:.4f} VaF1:{vf:.4f} EmaF1:{ef:.4f} {ema_tag}{star}')
    print(f'  Per-class: {" | ".join(f"C{i}:{f:.3f}" for i, f in enumerate(best_vp))}')

    history_log.append({'epoch': ep + 1, 'train_loss': tl, 'val_loss': vl, 'train_f1': tf, 'val_f1': vf, 'ema_f1': ef, 'per_class_f1': {CLASS_NAMES[i]: round(f, 4) for i, f in enumerate(best_vp)}, 'lr': lr_now, 'epoch_time': round(epoch_time, 1)})
    with open(f'{CFG.output_prefix}_history.json', 'w') as f:
        json.dump(history_log, f, indent=2)

    if epochs_no_improve >= CFG.early_stopping_patience:
        early_stop_epoch = ep + 1
        print(f'\n[EARLY STOP] No improvement for {CFG.early_stopping_patience} epochs. Best: {best_f1:.4f} at ep {best_ep}.')
        break

print(f'\n{"="*70}')
stop_reason = 'Early Stop' if early_stop_epoch else 'All epochs'
print(f'FOLD {FOLD_ID} Training done ({stop_reason}). Best Val F1: {best_f1:.4f} at epoch {best_ep}')
print(f'Total time: {(time.time() - GLOBAL_START) / 60:.1f} min')

## OOF Validation Inference

In [ ]:
# Load best model
print(f'\nLoading best model (epoch {best_ep}, F1={best_f1:.4f})...')
best_state = torch.load(f'{CFG.output_prefix}_best.pth', map_location=DEVICE)
model.load_state_dict(best_state)
model.eval()

print(f'\n--- OOF Validation Inference ---')
oof_ds = PigPostureValDataset(df_va, CFG.crops_dir, get_val_transforms(), cache=False)
oof_loader = DataLoader(oof_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

oof_probs_list, oof_labels_list, oof_row_ids_list = [], [], []
with torch.no_grad():
    for imgs, labels, row_ids in tqdm(oof_loader, desc='OOF Val'):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(imgs)
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        oof_probs_list.append(probs)
        oof_labels_list.extend(labels.numpy() if isinstance(labels, torch.Tensor) else labels)
        oof_row_ids_list.extend(row_ids)

oof_probs = np.concatenate(oof_probs_list)
oof_labels = np.array(oof_labels_list)
oof_row_ids = np.array(oof_row_ids_list, dtype=object)

np.save(f'{CFG.output_prefix}_oof_probs.npy', oof_probs)
np.save(f'{CFG.output_prefix}_oof_labels.npy', oof_labels)
np.save(f'{CFG.output_prefix}_oof_row_ids.npy', oof_row_ids)
oof_f1 = f1_score(oof_labels, oof_probs.argmax(1), average='macro')
print(f'Saved OOF: {CFG.output_prefix}_oof_probs.npy (shape: {oof_probs.shape})')
print(f'OOF Val Macro-F1: {oof_f1:.4f}')

gc.collect()
if DEVICE_TYPE == 'cuda':
    torch.cuda.empty_cache()

## Test Inference with TTA

In [ ]:
if len(df_test) > 0:
    print(f'\n--- Test Inference Fold {FOLD_ID} ({len(df_test)} images) ---')

    # Standard inference
    ids, all_probs = [], []
    with torch.no_grad():
        for imgs, row_ids in tqdm(test_loader, desc='Infer'):
            imgs = imgs.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(imgs)
            probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
            all_probs.append(probs)
            ids.extend(row_ids)

    test_probs = np.concatenate(all_probs)

    # TTA: Horizontal Flip with class 0<->1 swap
    print('TTA (Horizontal Flip with class swap)...')
    all_probs_tta = []
    with torch.no_grad():
        for imgs, _ in tqdm(test_loader, desc='TTA-HFlip'):
            imgs = imgs.to(DEVICE, non_blocking=True)
            imgs_flip = torch.flip(imgs, dims=[3])
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits_flip = model(imgs_flip)
            probs_flip = torch.softmax(logits_flip.float(), dim=1).cpu().numpy()
            probs_flip[:, [0, 1]] = probs_flip[:, [1, 0]]
            all_probs_tta.append(probs_flip)

    tta_probs = np.concatenate(all_probs_tta)
    final_probs = (test_probs + tta_probs) / 2.0
    final_preds = final_probs.argmax(axis=1)

    np.save(f'{CFG.output_prefix}_test_probs.npy', final_probs)
    np.save(f'{CFG.output_prefix}_test_row_ids.npy', np.array(ids, dtype=object))
    print(f'Saved: {CFG.output_prefix}_test_probs.npy (shape: {final_probs.shape})')

    # CSV submission
    sub_df = pd.DataFrame({'row_id': ids, 'class_id': final_preds})
    sub_path = f'T2_{CFG.output_prefix}_submission.csv'
    sub_df.to_csv(sub_path, index=False)
    dist = dict(zip(*np.unique(final_preds, return_counts=True)))
    print(f'Saved: {sub_path} | {len(sub_df)} rows | Dist: {dist}')
else:
    print('No test set found.')

## Summary

In [ ]:
print(f'\n===== FOLD {FOLD_ID} COMPLETE =====')
print(f'Total time: {(time.time() - GLOBAL_START) / 60:.1f} min')
print(f'\nFiles saved:')
print(f'  {CFG.output_prefix}_best.pth           (model checkpoint)')
print(f'  {CFG.output_prefix}_oof_probs.npy       (OOF validation probabilities)')
print(f'  {CFG.output_prefix}_oof_labels.npy      (OOF validation labels)')
print(f'  {CFG.output_prefix}_oof_row_ids.npy     (OOF validation row IDs)')
print(f'  {CFG.output_prefix}_test_probs.npy      (test probabilities with TTA)')
print(f'  {CFG.output_prefix}_test_row_ids.npy    (test row IDs)')
print(f'  {CFG.output_prefix}_history.json        (training metrics log)')
print(f'\n✅ Ready for ensemble with EVA02-Enormous predictions!')